# Fixture context — fixture difficulty (`fdr_avg`) and `total_points`

*Read-only informative artifact. This notebook characterises how fixture
difficulty changes the target picture so a human can decide how to read it. It
produces no gate decisions and no PROCEED/STOP verdict.*

## Questions a manager asks about fixture difficulty

- **Do players score more against easy fixtures than hard ones?** The whole
  premise of "fixture-chasing" — transferring players in for a good run — rests
  on this.
- **How big is the fixture effect?** A one-point swing between easy and hard is
  worth chasing; a fraction of a point is noise.
- **Does the effect differ by position?** Are attackers more fixture-sensitive
  than defenders, whose returns hinge on clean sheets?
- **Does easy-fixture scoring show up as fewer blanks, more returns, or both?**

Everything below is **season-pooled** over the study range and stratified by
fixture-difficulty tier.

## Setup

Load the mart, restrict to the **whole season** (GW 1 to the latest completed
GW) and the **participation** population (`minutes > 0`), then bin `fdr_avg`
into difficulty tiers.

This is a *descriptive characterisation* notebook, so it uses the full season —
no early-GW lower bound (that GW-6 cut in the older EDA-1 record was a
*predictive-evaluation* choice, not relevant here).

The population is everyone who **actually featured**: available players with
`minutes > 0`. This is a **participation** filter (the player appeared), **not
a performance gate**. `minutes` can be NULL for some rows; `minutes > 0`
naturally excludes those. The 60-minute performance-boundary question is
**deferred to the `population/` layer** and is deliberately not baked in here.

**FDR meaning.** `fdr_avg` is the average fixture difficulty rating for the
gameweek, range 1.0–5.0 (observed values 1, 2, 2.5, 3, 3.5, 4, 5), where **1 =
easiest** and **5 = hardest** opponent. For a DGW row it averages the two
fixtures' ratings, which is why half-step values appear.

**DGW pooling note.** This notebook pools SGW and DGW rows; a DGW row carries a
doubled `total_points`, so the per-tier means below are mildly DGW-inflated
where DGW rows land. DGW is rare (~1.5% of featured rows) and the dedicated
SGW-vs-DGW treatment lives in `dgw_vs_sgw.ipynb`; here `fdr_avg` is the axis of
interest.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display

from dal.pipeline import load as load_mart, run as run_pipeline
from dal.exceptions import MartNotBuiltError, MartSchemaError
from research.kernels.descriptive.distribution import (
    compute_distribution_stats,
    compare_cohorts,
)

try:
    _result = load_mart()
except (MartNotBuiltError, MartSchemaError) as _e:
    print(f"Rebuilding mart ({type(_e).__name__})...")
    run_pipeline(force=True)
    _result = load_mart()

# Descriptive characterisation uses the WHOLE season: GW 1 to the latest
# completed GW. No early-GW lower bound (that was a predictive-evaluation
# choice in the older EDA-1 record, not relevant here).
STUDY_GW_MIN = 1
STUDY_GW_MAX = _result.data_cutoff_gw

# Analytical population: PARTICIPATION filter, not a performance gate.
# Available players who actually featured -> minutes > 0. `minutes` can be NULL
# for some rows; minutes > 0 naturally excludes NULLs. The 60-minute
# performance boundary is NOT imposed here -- deferred to the population/ layer.
mart = _result.mart
df = mart[mart["gw"].between(STUDY_GW_MIN, STUDY_GW_MAX)].copy()
df = df[df["minutes"].notna() & (df["minutes"] > 0)].copy()
df = df[df["fdr_avg"].notna()].copy()  # FDR tier analysis needs a rating

POSITIONS = ["GK", "DEF", "MID", "FWD"]

# FDR tiers as OPERATIONAL HEURISTICS (not derived cut-points):
#   easy   : fdr_avg <= 2          (1 = easiest)
#   medium : 2 < fdr_avg < 4
#   hard   : fdr_avg >= 4          (5 = hardest)
TIER_ORDER = ["easy", "medium", "hard"]

def fdr_tier(v):
    if v <= 2:
        return "easy"
    if v >= 4:
        return "hard"
    return "medium"

df["fdr_tier"] = df["fdr_avg"].map(fdr_tier)
df["fdr_tier"] = pd.Categorical(df["fdr_tier"], categories=TIER_ORDER, ordered=True)

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 140)
pd.set_option("display.float_format", "{:.3f}".format)

print(f"Study range: GW {STUDY_GW_MIN} - GW {STUDY_GW_MAX} (whole season, from mart data_cutoff_gw)")
print(f"Population: minutes > 0 (participation, not a performance gate), with fdr_avg present, n = {len(df):,}")
print("\nFDR tiers (operational heuristics):  easy = fdr_avg <= 2 | medium = 2 < fdr_avg < 4 | hard = fdr_avg >= 4")
print("Tier counts:")
for tier in TIER_ORDER:
    print(f"  {tier:<7}: {int((df['fdr_tier'] == tier).sum()):>6,}")

## (a) `total_points` distribution by FDR tier and position

**What we measure** — the distribution of `total_points` (count, mean, median,
spread, p90) across the three FDR tiers — easy / medium / hard — **within each
position**, via `compare_cohorts` with the tiers as cohorts. The cutoffs
(`easy = fdr_avg <= 2`, `medium = 2 < fdr_avg < 4`, `hard = fdr_avg >= 4`) are
stated **operational heuristics**, not derived breakpoints.

*Stat glossary for this table:* **mean** — arithmetic average, pulled up by rare hauls; **median** — middle value, the robust "typical" return; **std** — typical distance from the mean in points, sensitive to outliers; **IQR (p75−p25)** — spread of the middle 50%, robust to skew; **p90/p99** — score exceeded by only 10%/1% of appearances (practical ceiling); **skew** — positive = long right tail of rare high scores.

**What it means** — this is the direct read on whether **fixture-chasing is
justified by the data**: the easy-minus-hard gap in mean/median `total_points`
is the points a manager can expect to gain by timing transfers into a soft run,
position by position. A wide gap for attackers and a narrow one for keepers, say,
would tell a manager where fixture-chasing pays.

**What it doesn't mean** — this is **association, descriptively shown**, not a
causal or predictive claim: easy-fixture rows also concentrate strong teams
(good sides draw easier-looking runs), so part of any gap is team quality, not
the fixture alone. The relationship-strength / conditional-RHO question is
**out of scope** for this descriptive layer. Tiers are coarse heuristic bins,
and figures are **season-pooled** (DGW rows included, mildly inflating tiers
where they land).

**Guiding question** — *Do players score more against easy fixtures than hard
ones, and how big is the gap by position?*

In [ ]:
tier_rows = []
for pos in POSITIONS:
    sub = df[df.position == pos]
    cohorts = {tier: sub[sub["fdr_tier"] == tier] for tier in TIER_ORDER}
    stats = compare_cohorts(cohorts, value_col="total_points")
    for tier in TIER_ORDER:
        s = stats.loc[tier]
        tier_rows.append({
            "position": pos,
            "fdr_tier": tier,
            "n": int(s["count"]) if not np.isnan(s["count"]) else 0,
            "mean": s["mean"],
            "median": s["median"],
            "std": s["std"],
            "p90": s["p90"],
        })
fdr_by_pos = pd.DataFrame(tier_rows)
display(fdr_by_pos.round(2))

# Easy-minus-hard mean gap per position -- the headline fixture-chasing number.
gap_rows = []
for pos in POSITIONS:
    p = fdr_by_pos[fdr_by_pos.position == pos].set_index("fdr_tier")
    gap_rows.append({
        "position": pos,
        "mean_easy": p.loc["easy", "mean"],
        "mean_hard": p.loc["hard", "mean"],
        "easy_minus_hard": round(p.loc["easy", "mean"] - p.loc["hard", "mean"], 2),
    })
fdr_gap = pd.DataFrame(gap_rows)
display(fdr_gap.round(2))

In [ ]:
# Mean and median Y by tier per position reveal the gradient a table flattens:
# a downward slope easy -> hard is the fixture effect, position by position.
colours = {"GK": "#9467bd", "DEF": "#1f77b4", "MID": "#2ca02c", "FWD": "#d62728"}
x = np.arange(len(TIER_ORDER))
fig, (ax_mean, ax_med) = plt.subplots(1, 2, figsize=(13, 4), sharex=True)
for pos in POSITIONS:
    sub = fdr_by_pos[fdr_by_pos.position == pos].set_index("fdr_tier").reindex(TIER_ORDER)
    ax_mean.plot(x, sub["mean"].to_numpy(dtype=float), "-o", color=colours[pos], label=pos)
    ax_med.plot(x, sub["median"].to_numpy(dtype=float), "-o", color=colours[pos], label=pos)
for ax, title in [(ax_mean, "mean total_points"), (ax_med, "median total_points")]:
    ax.set_xticks(x)
    ax.set_xticklabels(TIER_ORDER)
    ax.set_xlabel("FDR tier (easy -> hard)")
    ax.set_title(title, fontsize=10)
    ax.legend(fontsize=8)
ax_mean.set_ylabel("total_points")
fig.suptitle("total_points by FDR tier per position (minutes > 0, SGW+DGW pooled)", y=1.02)
plt.tight_layout()
plt.show()

## (b) Blank rate and return rate by FDR tier and position

**What we measure** — within each (position, FDR tier) cell, the share of
player-gameweeks that **blank** (`total_points == 0`) and the share that
**return** (`total_points >= 4`). Computed inline as simple proportions.

**What it means** — this decomposes the mean gap from (a) into *mechanism*: does
an easy fixture help by **avoiding blanks**, by **producing more returns**, or
both? If easy fixtures mainly cut the blank rate, fixture-chasing is about floor
(fewer dud weeks); if they lift the return rate, it is about ceiling (more
hauls). The 4+ line is the standard "did something useful" bar.

**What it doesn't mean** — these are **season-pooled** rates over the
**participation** population, so a blank here includes brief cameos that
returned nothing (no 60-minute gate — that is deferred to `population/`). The
same team-quality confound as (a) applies: easy-fixture rows skew toward
stronger sides. Rates are descriptive association, not predictive probabilities
for a specific player-week.

**Guiding question** — *Does easy-fixture scoring show up as fewer blanks, more
returns, or both — and does the channel differ by position?*

In [ ]:
fdr_rates = (
    df.groupby(["position", "fdr_tier"])["total_points"]
    .apply(lambda y: pd.Series({
        "n": len(y),
        "blank_rate_%": round((y == 0).mean() * 100, 1) if len(y) else float("nan"),
        "return_4+_%": round((y >= 4).mean() * 100, 1) if len(y) else float("nan"),
    }))
    .reset_index()
)
display(fdr_rates)


## What fixture difficulty looks like

Plain-language summary of the descriptive picture (not a verdict):

- The tier table and the gradient plot (a) show, position by position, whether
  `total_points` slopes downward from easy to hard fixtures, and the
  **easy-minus-hard mean gap** sizes how much fixture-chasing could be worth.
- The blank/return table (b) shows the *channel*: whether an easy fixture helps
  mainly by cutting blanks (floor) or lifting 4+ returns (ceiling), and whether
  that channel differs across positions.
- Any gap is **descriptive association**, not causation: easy-fixture rows skew
  toward stronger teams, so team quality is baked into the gradient. The
  tier cutoffs are **operational heuristics**, and rates use the
  **participation** population (cameos included).

All figures are **whole-season**, season-pooled, over the **participation**
population (`minutes > 0` — not a performance gate; the 60-minute boundary is
deferred to the `population/` layer). DGW-vs-SGW and home/away are treated in
the sibling notebooks.